# CAMI III benchmark

CDS-level performance on the CAMI III simulated metagenome (sample_0), with the CAMI genomes that overlap the
DeepCDS training/validation genomes identified and removed.

* `01_per_bin_counts.py`: TP/FP/FN per genome bin and model on one CAMI test set variant (`VARIANT` below; one
  run per variant with `--variant`). Every read is attributed to its genome bin, so a test set without some
  genomes = the same counts without their bins.
* `02_genome_overlap.py`: for each CAMI genome bin, the closest train/val genome by ANI (skani) and whether any
  of its contigs belongs to a train/val assembly. Categories: **identical** (same assembly or ANI >= 99.9%),
  **same species** (ANI >= 95%), **novel** (below), **non-prokaryotic** (Eukaryota / Viruses; cannot overlap
  with the prokaryotic training set).

Metrics are pooled over all reads of the included bins; SE = genome-level cluster bootstrap over bins
(N = 1000, seed 42). Section 5 makes the model comparison table: three test set variants, each without the
identical and same-species genomes.

In [ ]:
import os

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

RESULTS_DIR = "/tmp/nrt204/FragmentPredictor/data/processed_data/cami_main_benchmark"
SAVE_OUTPUTS = False      # True: write tables (csv, tex) to RESULTS_DIR/tables/

# CAMI test set variant for sections 1-3 (per-bin counts written by 01_per_bin_counts.py --variant <VARIANT>):
# "CAMI_metagenome" (base), "CAMI_metagenome_no_ab_initio" (excluding uncertain annotations),
# "CAMI_metagenome_no_cds_contigs" (excluding contigs lacking annotations) or
# "CAMI_metagenome_no_ab_initio_no_cds_contigs" (both). Section 5 uses the first three regardless of this setting.
VARIANT = "CAMI_metagenome_no_ab_initio_no_cds_contigs"

MODELS = {"fgs_complete": "FGS (Complete)", "fgs_illumina_5": "FGS (0.5% error rate)",
          "fgs_illumina_10": "FGS (1.0% error rate)", "prodigal": "MetaProdigal",
          "deep_cds_no_errors": "DeepCDS N", "deep_cds_substitution": "DeepCDS S", "deep_cds_errors": "DeepCDS S+I"}
TEST_SETS = {
    "Full CAMI set": [],
    "Without identical genomes": ["identical"],
    "Without identical and same-species genomes": ["identical", "same species"],
}
N_BOOTSTRAP, SEED = 1000, 42


def counts_file(variant):
    return (f"{RESULTS_DIR}/per_bin_counts.tsv" if variant == "CAMI_metagenome_no_ab_initio_no_cds_contigs"
            else f"{RESULTS_DIR}/per_bin_counts_{variant}.tsv")


def load_counts(variant):
    c = pd.read_csv(counts_file(variant), sep="\t")
    c["category"] = c["genome_bin_id"].map(genomes.set_index("genome_bin_id")["category"]).fillna("unmapped bin")
    assert (c["category"] != "unmapped bin").all()
    return c


def save_table(table, name):
    if SAVE_OUTPUTS:
        os.makedirs(f"{RESULTS_DIR}/tables", exist_ok=True)
        table.to_csv(f"{RESULTS_DIR}/tables/{name}.csv")

In [ ]:
genomes = pd.read_csv(f"{RESULTS_DIR}/cami_genome_overlap.tsv", sep="\t")
genomes["genome"] = genomes["source_file"].map(lambda p: os.path.basename(p).replace("_genomic.fna", "") if isinstance(p, str) else p)
counts = load_counts(VARIANT)
print(f"Test set variant: {VARIANT}")

## 1. Overlap between CAMI genomes and the training/validation genomes

In [ ]:
ref = counts[counts["model"] == "deep_cds_no_errors"]
labeled = (ref["tp"] + ref["fn"]).groupby(ref["category"]).sum()
summary = pd.DataFrame({
    "genome bins (all)": genomes["category"].value_counts(),
    "genome bins in this test set": ref.groupby("category")["genome_bin_id"].nunique(),
    "labeled CDS": labeled,
    "fraction of labeled CDS": (labeled / labeled.sum()).round(4),
}).reindex(["identical", "same species", "novel", "non-prokaryotic"])
display(summary)
save_table(summary, "overlap_summary")

cols = ["genome_bin_id", "genome", "ANI", "Align_fraction_query", "Align_fraction_ref", "train_genome",
        "train_species", "train_partition", "same_assembly_as"]
print("Identical and same-species genomes (closest train/val genome by ANI):")
overlapping = genomes[genomes["category"].isin(["identical", "same species"])].sort_values("ANI", ascending=False)[cols]
display(overlapping)
save_table(overlapping, "overlapping_genomes")

In [ ]:
prok = genomes[genomes["domain"].isin(["Bacteria", "Archaea"])]
print(f"{len(prok)} prokaryotic CAMI genomes; best ANI to any train/val genome:")
display(pd.cut(prok["ANI"].fillna(0), [0, 80, 90, 95, 99, 99.9, 100.001], right=False,
               labels=["<80% (no skani hit)", "80-90%", "90-95%", "95-99%", "99-99.9%", ">=99.9%"]).value_counts().sort_index())

## 2. Performance on stricter test sets

In [ ]:
def pooled(tp, fp, fn):
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
    r = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0)
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(tp, dtype=float), where=(p + r) > 0)
    return {"precision": p, "recall": r, "f1_score": f1}


def metrics_with_se(sub, rng):
    tp, fp, fn = (sub[k].to_numpy(dtype=float) for k in ("tp", "fp", "fn"))
    point = {k: float(v) for k, v in pooled(np.array([tp.sum()]), np.array([fp.sum()]), np.array([fn.sum()])).items()}
    idx = rng.integers(0, len(tp), size=(N_BOOTSTRAP, len(tp)))   # resample genome bins with replacement
    boot = pooled(tp[idx].sum(axis=1), fp[idx].sum(axis=1), fn[idx].sum(axis=1))
    return point, {k: float(v.std(ddof=1)) for k, v in boot.items()}


tables = {}
for name, excluded in TEST_SETS.items():
    rng = np.random.default_rng(SEED)
    rows = {}
    for model, label in MODELS.items():
        sub = counts[(counts["model"] == model) & ~counts["category"].isin(excluded)]
        point, se = metrics_with_se(sub, rng)
        rows[label] = {m: f"{point[m]:.3f} ± {se[m]:.3f}" for m in ("precision", "recall", "f1_score")}
    ref = counts[(counts["model"] == "deep_cds_no_errors") & ~counts["category"].isin(excluded)]
    print(f"=== {name}: {ref['genome_bin_id'].nunique()} genome bins, {int((ref['tp'] + ref['fn']).sum()):,} labeled CDS")
    tables[name] = pd.DataFrame(rows).T
    display(tables[name])
    save_table(tables[name], "metrics_" + name.lower().replace(" ", "_").replace("-", "_"))

## 3. Performance by overlap category

In [ ]:
rows = []
for cat in ["same species", "novel", "non-prokaryotic"]:
    rng = np.random.default_rng(SEED)
    for model, label in MODELS.items():
        sub = counts[(counts["model"] == model) & (counts["category"] == cat)]
        point, se = metrics_with_se(sub, rng)
        rows.append({"category": cat, "model": label,
                     **{m: f"{point[m]:.3f} ± {se[m]:.3f}" for m in ("precision", "recall", "f1_score")}})
by_category = (pd.DataFrame(rows).pivot(index="model", columns="category", values="f1_score")
               .loc[list(MODELS.values()), ["same species", "novel", "non-prokaryotic"]])
ref = counts[counts["model"] == "deep_cds_no_errors"]
by_category.columns = [f"{c} ({ref.loc[ref['category'] == c, 'genome_bin_id'].nunique()} bins)" for c in by_category.columns]
print("F1 by category (identical genomes: 2 bins, too few for a bootstrap):")
display(by_category)
save_table(pd.DataFrame(rows), "metrics_by_category")

## 4. Fragment-level sequence identity to the training data

Every labeled CAMI CDS fragment was searched (MMseqs2, protein level, `-s 7.5 -e 10 --min-aln-len 20`)
against the same train/val CDS fragment database as the homology audit, and binned by the identity of its
best (max-bitscore) hit (`03`-`05`). *No hit* = no alignment of >= 20 aa to any train/val fragment
(fragments < 20 aa were not searched and count as no hit). TP = exact match, as in the CAMI benchmark.

In [ ]:
BIN_ORDER = ["no hit", "<30%", "30-50%", "50-70%", "70-90%", ">=90%"]
LENGTH_LABELS = ["20-29 aa", "30-39 aa", "40-49 aa", ">=50 aa"]
MAIN_MODELS = ["deep_cds_no_errors", "deep_cds_substitution", "deep_cds_errors", "prodigal", "fgs_complete"]

frag_counts = pd.read_csv(f"{RESULTS_DIR}/cami_fragment_counts.tsv", sep="\t")
frag_counts["category"] = frag_counts["genome_bin_id"].map(genomes.set_index("genome_bin_id")["category"])
ref = frag_counts[frag_counts["model"] == "deep_cds_no_errors"]
dist = ref.groupby("bin")["n"].sum().reindex(BIN_ORDER).to_frame("labeled CDS")
dist["fraction"] = (dist["labeled CDS"] / dist["labeled CDS"].sum()).round(4)
for cat in ["same species", "novel", "non-prokaryotic"]:
    sub = ref[ref["category"] == cat].groupby("bin")["n"].sum().reindex(BIN_ORDER).fillna(0)
    dist[f"fraction, {cat} genomes"] = (sub / sub.sum()).round(4)
display(dist)
save_table(dist, "fragment_identity_distribution")

In [ ]:
def recall_by(sub_counts, group_col, groups, models=MAIN_MODELS):
    # Pooled recall per group with genome-bin cluster bootstrap SE; columns labelled with n
    out = {}
    for grp in groups:
        g = sub_counts[sub_counts[group_col] == grp]
        rng = np.random.default_rng(SEED)
        col = {}
        for model in models:
            per_bin = g[g["model"] == model].groupby("genome_bin_id")[["tp", "n"]].sum()
            if per_bin["n"].sum() == 0:
                continue
            tp, n = per_bin["tp"].to_numpy(float), per_bin["n"].to_numpy(float)
            idx = rng.integers(0, len(tp), size=(N_BOOTSTRAP, len(tp)))
            se = (tp[idx].sum(axis=1) / n[idx].sum(axis=1)).std(ddof=1)
            col[MODELS[model]] = f"{tp.sum() / n.sum():.3f} ± {se:.3f}"
        n_grp = int(g[g["model"] == models[0]]["n"].sum())
        out[f"{grp} (n={n_grp:,})"] = col
    return pd.DataFrame(out)


print("Recall per identity bin (all genome bins):")
recall_bins = recall_by(frag_counts, "bin", BIN_ORDER)
display(recall_bins)
save_table(recall_bins, "recall_per_identity_bin")

print("Recall per identity bin, novel genomes only (ANI < 95% to every train/val genome):")
display(recall_by(frag_counts[frag_counts["category"] == "novel"], "bin", BIN_ORDER))

In [ ]:
print("Recall on 'no hit' vs hit fragments within length classes:")
fc = frag_counts.assign(hit=np.where(frag_counts["bin"] == "no hit", "no hit", "hit"))
for cls in LENGTH_LABELS:
    sub = fc[fc["length_class"] == cls]
    if sub.empty:
        continue
    print(f"--- {cls} (n={int(sub[sub['model'] == 'deep_cds_no_errors']['n'].sum()):,})")
    display(recall_by(sub, "hit", ["no hit", "hit"]))

## 5. Model comparison table: three test set variants without overlapping genomes

CDS-level F1, sensitivity and precision on the base CAMI test set, the set excluding reads overlapping uncertain
annotations (hypothetical or ab initio predicted) and the set excluding contigs without any annotations, each
without the CAMI genomes that are identical or the same species (ANI >= 95%) as a training/validation genome.
SE = genome-level cluster bootstrap (as in section 2). Best value per metric and block in bold.
Requires `01_per_bin_counts.py --variant` for `CAMI_metagenome`, `CAMI_metagenome_no_ab_initio` and
`CAMI_metagenome_no_cds_contigs`.

In [ ]:
TABLE_BLOCKS = {"Base": "CAMI_metagenome",
                "Excluding uncertain annotations": "CAMI_metagenome_no_ab_initio",
                "Excluding contigs lacking annotations": "CAMI_metagenome_no_cds_contigs"}
EXCLUDED = ["identical", "same species"]
METRICS = [("f1_score", "F1 Score"), ("recall", "Sensitivity"), ("precision", "Precision")]

lines = [r"\begin{table}[ht]", r"\centering", r"\begin{tabular}{lccc}", r"\toprule",
         "& " + " & ".join(label for _, label in METRICS) + r" \\"]
for block, variant in TABLE_BLOCKS.items():
    c = load_counts(variant)
    c = c[~c["category"].isin(EXCLUDED)]
    rng = np.random.default_rng(SEED)
    res = {}
    for model, label in MODELS.items():
        res[label] = metrics_with_se(c[c["model"] == model], rng)
    n_bins = c.loc[c["model"] == "deep_cds_no_errors", "genome_bin_id"].nunique()
    print(f"{block} ({variant}): {n_bins} genome bins")
    best = {m: max(round(res[l][0][m], 3) for l in res) for m, _ in METRICS}
    lines += [r"\midrule", rf"\multicolumn{{4}}{{l}}{{\textbf{{{block}}}}} \\", r"\midrule"]
    for label, (point, se) in res.items():
        cells_ = []
        for m, _ in METRICS:
            v = f"{point[m]:.3f} $\pm$ {se[m]:.3f}"
            cells_.append(rf"\textbf{{{v}}}" if round(point[m], 3) == best[m] else v)
        lines.append(label.replace("%", r"\%") + " & " + " & ".join(cells_) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\label{tab:cds_metagenome_model_comparison}", r"\end{table}"]
latex = "\n".join(lines)
print(latex)
if SAVE_OUTPUTS:
    os.makedirs(f"{RESULTS_DIR}/tables", exist_ok=True)
    with open(f"{RESULTS_DIR}/tables/cami_model_comparison.tex", "w") as fh:
        fh.write(latex + "\n")
